# Phase 4: Anomaly Scoring Features

**Produces:** `anomaly_scores.parquet` — 2 unsupervised anomaly signals per claim

| Feature | Method | Why |
|---------|--------|-----|
| `dae_recon_error` | Denoising Autoencoder | Reconstruction error = how unusual a claim is. Porto Seguro 1st place used this. |
| `isoforest_score` | Isolation Forest | Anomaly score from tree-based path length. IEEE paper: +0.01-0.02 PR-AUC on insurance fraud. |

**Design rules:**
- Both models trained on TRAIN data only (no leakage)
- Score computed on ALL rows (train + test)
- Numeric features only (after imputation)
- Final output row-aligned with `CFM_anon_final.csv`

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
import warnings, time, os
warnings.filterwarnings('ignore')

tf.random.set_seed(42)
np.random.seed(42)

DATA_PATH  = r"C:\Users\mitul\OneDrive\Documents\arya\fraud\CFM_anon_final.csv"
OUT_PATH   = r"C:\Users\mitul\OneDrive\Documents\arya\fraud\anomaly_scores.parquet"
SPLIT_DATE = pd.Timestamp('2025-07-01')

print('Libraries loaded')

In [ ]:
# ── Load + Split ─────────────────────────────────────────────────────────────
t0 = time.time()
df = pd.read_csv(DATA_PATH, low_memory=False)
df['data_created_at_parsed'] = pd.to_datetime(df['data_created_at'], format='ISO8601')
print(f"Loaded {len(df):,} rows x {len(df.columns)} cols | {time.time()-t0:.1f}s")

train_mask = df['data_created_at_parsed'] < SPLIT_DATE
test_mask  = ~train_mask
print(f"Train: {train_mask.sum():,}  Test: {test_mask.sum():,}")

In [ ]:
# ── Feature prep: numeric only, impute with train median ─────────────────────
# Drop ID/label/date cols — only want signal features for anomaly detection
DROP_COLS = [
    # Row identifiers
    'Claim_No', 'data_created_at', 'data_created_at_parsed',
    # TARGET LABELS — must drop to prevent leakage
    'Target_as_investigation', 'Target_as_fraud',
    'Investigated', 'Old_Target',
    # Post-adjudication columns (set AFTER claim is decided — leaky)
    'Claims_Stage',           # reflects processing stage
    'Claim_Reserve',          # set by adjuster after investigation
    'Approved_Amount_INR',    # set after adjudication (fraud→lower approval)
    'Buffer_Available',       # derived from approved amount
    'Buffer_Consumed',        # derived from approved amount
    'Balance_Buffer',         # derived from approved amount
    'Investigation Outcome', 'Fraud_Outcome',
    'Assign Date', 'Case_Close_Date',
    # Date strings
    'Company_Date_of_Joining', 'Date_of_Joining_the_Policy',
    'Risk_Inception_Date', 'Risk_Expiry_Date',
    'Expected_Date_of_Admission', 'Expected_Date_of_Discharge',
    'Actual_Date_of_Admission', 'Actual_Date_of_Discharge',
    'Document_Received_Date', 'Claim_Intimation_Date',
    'Medical_Management_Date', 'Last_Document_Received_Date',
    # High-cardinality ID strings
    'HID_anon', 'CID_anon', 'Policy_number', 'DID_anon',
    'Final_Diagnosis', 'Claim_Type', 'Insurer',
]

df_feat = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')

# Keep only numeric columns
num_cols = df_feat.select_dtypes(include=['int64','float64','int32','float32']).columns.tolist()
print(f"Numeric feature cols: {len(num_cols)}")
print("Columns used:")
for c in num_cols:
    print(f"  {c}")
X_all = df_feat[num_cols].copy()

# Impute with train-set median (no leakage)
# replace([inf,-inf], Series) fails — convert inf->NaN first, then fillna
train_medians = X_all.loc[train_mask].median()
X_all = X_all.replace([np.inf, -np.inf], np.nan)
X_all = X_all.fillna(train_medians)
X_all = X_all.fillna(0)  # fallback for cols with all-NaN in train

print(f"\nX_all shape: {X_all.shape} | Missing after impute: {X_all.isnull().sum().sum()}")

# Clip extreme values before scaling: cap at 1st/99th percentile per column
# (prevents near-zero IQR cols from blowing up RobustScaler output)
tr_p01 = X_all.loc[train_mask].quantile(0.01)
tr_p99 = X_all.loc[train_mask].quantile(0.99)
X_all = X_all.clip(lower=tr_p01, upper=tr_p99, axis=1)

# Scale with train stats only
scaler = RobustScaler()
X_tr_sc = scaler.fit_transform(X_all.loc[train_mask])
X_all_sc = scaler.transform(X_all)
print(f"Train scaled: {X_tr_sc.shape}  |  All scaled: {X_all_sc.shape}")
print(f"X_tr_sc abs max: {np.abs(X_tr_sc).max():.2f}  mean: {np.abs(X_tr_sc).mean():.4f}")

In [ ]:
# ── Denoising Autoencoder (DAE) ───────────────────────────────────────────────
# Architecture: Porto Seguro 1st place style
#   Encoder: input -> 128 -> 64 -> 32 (bottleneck)
#   Decoder: 32 -> 64 -> 128 -> input
# Training: add Gaussian noise to input, reconstruct clean input
# Signal:  reconstruction error on TEST data = how unusual the claim is

n_features = X_tr_sc.shape[1]
NOISE_FACTOR = 0.1
EPOCHS = 50
BATCH  = 512

def build_dae(n_in):
    inp = keras.Input(shape=(n_in,))
    # Add noise during training
    noisy = layers.GaussianNoise(NOISE_FACTOR)(inp)
    # Encoder
    x = layers.Dense(128, activation='relu')(noisy)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(32, activation='relu')(x)  # bottleneck
    # Decoder
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(n_in, activation='linear')(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return model

print(f"Building DAE: {n_features} -> 128 -> 64 -> 32 -> 64 -> 128 -> {n_features}")
dae = build_dae(n_features)

t = time.time()
cb = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor='val_loss', verbose=0),
]
hist = dae.fit(
    X_tr_sc, X_tr_sc,        # input=noisy (GaussianNoise layer), target=clean
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=0,
)
best_epoch = np.argmin(hist.history['val_loss']) + 1
print(f"DAE trained: {best_epoch} epochs | val_loss={min(hist.history['val_loss']):.5f} | {time.time()-t:.1f}s")

# Reconstruction error on all data (train + test)
X_recon = dae.predict(X_all_sc, batch_size=2048, verbose=0)
dae_error = np.mean((X_all_sc - X_recon) ** 2, axis=1)  # MSE per row
print(f"DAE recon error — train mean: {dae_error[train_mask].mean():.4f}  test mean: {dae_error[test_mask].mean():.4f}")
print(f"  (higher error on test = distribution shift captured)")

In [ ]:
# ── Isolation Forest ──────────────────────────────────────────────────────────
# Trained on TRAIN set only.
# score_samples() returns negative anomaly score: lower = more anomalous.
# We negate so higher = more anomalous (consistent with dae_error direction).

print("Training Isolation Forest...")
t = time.time()
iso = IsolationForest(
    n_estimators=200,
    max_samples='auto',
    contamination=0.05,   # ~5% expected anomaly rate
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_tr_sc)
print(f"  IsoForest trained | {time.time()-t:.1f}s")

t = time.time()
iso_score_raw = iso.score_samples(X_all_sc)   # lower = more anomalous
iso_score     = -iso_score_raw                 # negate: higher = more anomalous
print(f"  Scored all rows | {time.time()-t:.1f}s")
print(f"IsoForest score — train mean: {iso_score[train_mask].mean():.4f}  test mean: {iso_score[test_mask].mean():.4f}")

In [ ]:
# ── Quick fraud-label sanity check ────────────────────────────────────────────
# If anomaly scores correlate with fraud labels in train, they're meaningful features.
from sklearn.metrics import roc_auc_score

# Fraud target
fraud_col = None
for c in ['Fraud_Outcome', 'Fraud', 'fraud']:
    if c in df.columns:
        fraud_col = c; break

inv_col = None
for c in ['Investigation Outcome', 'inv_outcome']:
    if c in df.columns:
        inv_col = c; break

if fraud_col:
    y_fraud_tr = pd.to_numeric(df.loc[train_mask, fraud_col], errors='coerce').fillna(0).astype(int)
    if y_fraud_tr.sum() > 0:
        auc_dae = roc_auc_score(y_fraud_tr, dae_error[train_mask])
        auc_iso = roc_auc_score(y_fraud_tr, iso_score[train_mask])
        print(f"Fraud ROC-AUC on TRAIN — DAE error: {auc_dae:.4f}  |  IsoForest: {auc_iso:.4f}")
        print(f"  (>0.55 = meaningful unsupervised signal)")

if inv_col:
    y_inv_tr = (df.loc[train_mask, inv_col].astype(str).str.lower().isin(['1','yes','true','investigated'])).astype(int)
    if y_inv_tr.sum() > 0:
        auc_dae_i = roc_auc_score(y_inv_tr, dae_error[train_mask])
        auc_iso_i = roc_auc_score(y_inv_tr, iso_score[train_mask])
        print(f"Inv   ROC-AUC on TRAIN — DAE error: {auc_dae_i:.4f}  |  IsoForest: {auc_iso_i:.4f}")

In [ ]:
# ── Save anomaly_scores.parquet ───────────────────────────────────────────────
out = pd.DataFrame({
    'dae_recon_error': dae_error.astype(np.float32),
    'isoforest_score': iso_score.astype(np.float32),
}, index=df.index)

out.to_parquet(OUT_PATH)
print(f"Saved: {OUT_PATH}")
print(f"  Shape: {out.shape}")
print(f"  dae_recon_error  — min={out.dae_recon_error.min():.4f}  max={out.dae_recon_error.max():.4f}  mean={out.dae_recon_error.mean():.4f}")
print(f"  isoforest_score  — min={out.isoforest_score.min():.4f}  max={out.isoforest_score.max():.4f}  mean={out.isoforest_score.mean():.4f}")

# Verify row alignment
assert len(out) == len(df), "Row count mismatch!"
print("Row alignment OK.")